# Ablation 2, Variant C: SmallMamba + Continuous Head (MSE)

**Purpose**: Confirm that Mamba's geometric stability survives even when we
strip out the discrete categorical head and replace it with a continuous MSE
regression. This maintains perfect parity with Variant A (SmallBERT + MSE).

### Critical Design: Dual-Return Generators

Uses raw continuous ODE floats as MSE targets (NOT requantized tokens).
Input: discrete 256-bin tokens. Target: raw continuous [0,1] values.

**Expected Result**: Mamba remains perfectly stable. The ODE prior is
architecture-intrinsic, not loss-function-dependent.

## Prerequisites
- Upload `evaluation_harness.py` to `/content/`
- GPU runtime required
- Run after baseline `Architectural_Control_Experiment.ipynb`

---

In [ ]:
# Dependencies
print('Installing dependencies...')
!pip install -q torch shesha-geometry matplotlib seaborn pandas scipy
print('Building mamba-ssm...')
!pip install causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -5

MAMBA_AVAILABLE = False
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm: OK')
except ImportError:
    print('mamba-ssm: FAILED -- using pure-PyTorch fallback')

import torch
print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Configuration
import os, sys, gc, time
import numpy as np
import pandas as pd
sys.path.insert(0, '.')

OUTPUT_BASE = './results/ablation2_variant_c_mamba_continuous/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'
for d in [RESULTS_DIR, CACHE_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

SEQ_LEN = 512; N_BINS = 256; N_TRAIN = 50_000; N_EVAL = 2_000; SEED = 320
PAD_TOKEN = N_BINS; MASK_TOKEN = N_BINS+1; VOCAB_SIZE = N_BINS+2
D_MODEL = 256; N_LAYERS = 4; N_HEADS = 4; FFN_DIM = 1024
D_STATE = 16; D_CONV = 4; EXPAND = 2
LR = 3e-4; WEIGHT_DECAY = 0.01; EPOCHS = 20; BATCH_SIZE = 64
NOISE_RATES = [0.01, 0.02, 0.05, 0.10]
DATASETS = ['waveform', 'oscillator', 'lorenz']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}\nAblation 2C: SmallMamba + Continuous MSE Head (raw ODE targets)')

In [ ]:
# DUAL-RETURN Dataset Generators
#
# Returns (discrete_tokens, continuous_values) to avoid staircase leakage.

from scipy.integrate import solve_ivp

def discretize(values, n_bins=N_BINS, global_min=None, global_max=None):
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12: return np.full_like(values, n_bins//2, dtype=np.int64)
    normed = np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0)
    return np.clip((normed * (n_bins-1)).astype(np.int64), 0, n_bins-1)

def normalize_to_01(values, global_min=None, global_max=None):
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12: return np.full_like(values, 0.5, dtype=np.float32)
    return ((values - vmin) / (vmax - vmin)).astype(np.float32)

def generate_waveforms(n, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 1, seq_len, endpoint=False)
    raw = []
    for _ in range(n):
        s = sum(rng.uniform(0.1,1)*np.sin(2*np.pi*rng.uniform(0.5,50)*t+rng.uniform(0,2*np.pi)) for _ in range(rng.integers(3,6)))
        raw.append(s)
    if global_range is None:
        av = np.concatenate(raw); global_range = (float(av.min()), float(av.max()))
    gn, gx = global_range
    return (np.array([discretize(s, global_min=gn, global_max=gx) for s in raw], dtype=np.int64),
            np.array([normalize_to_01(s, global_min=gn, global_max=gx) for s in raw], dtype=np.float32), global_range)

def generate_oscillators(n, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 4, seq_len, endpoint=False)
    raw = [rng.uniform(0.5,2)*np.exp(-rng.uniform(0.2,2)*t)*np.cos(rng.uniform(2,20)*t+rng.uniform(0,2*np.pi)) for _ in range(n)]
    if global_range is None:
        av = np.concatenate(raw); global_range = (float(av.min()), float(av.max()))
    gn, gx = global_range
    return (np.array([discretize(s, global_min=gn, global_max=gx) for s in raw], dtype=np.int64),
            np.array([normalize_to_01(s, global_min=gn, global_max=gx) for s in raw], dtype=np.float32), global_range)

def _lorenz_rhs(t, state, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x, y, z = state
    return [sigma*(y-x), x*(rho-z)-y, x*y-beta*z]

def generate_lorenz(n, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    from scipy.integrate import solve_ivp
    rng = np.random.default_rng(seed)
    t_span, t_eval = (0, 25), np.linspace(0, 25, seq_len)
    raw = []
    for _ in range(n):
        x0,y0,z0 = rng.uniform(-15,15), rng.uniform(-15,15), rng.uniform(10,40)
        sol = solve_ivp(_lorenz_rhs, t_span, [x0,y0,z0], t_eval=t_eval, method='RK45', max_step=0.05)
        if sol.success and len(sol.y[0]) == seq_len: raw.append(sol.y[0])
        else:
            sol2 = solve_ivp(_lorenz_rhs, t_span, [x0+rng.uniform(-1,1),y0,z0], t_eval=t_eval, method='RK45', max_step=0.01)
            s = sol2.y[0][:seq_len]
            if len(s) < seq_len: s = np.pad(s, (0, seq_len-len(s)), mode='edge')
            raw.append(s)
    if global_range is None:
        av = np.concatenate(raw); global_range = (float(av.min()), float(av.max()))
    gn, gx = global_range
    return (np.array([discretize(s, global_min=gn, global_max=gx) for s in raw], dtype=np.int64),
            np.array([normalize_to_01(s, global_min=gn, global_max=gx) for s in raw], dtype=np.float32), global_range)

GLOBAL_RANGES = {}
GENERATORS = {'waveform': generate_waveforms, 'oscillator': generate_oscillators, 'lorenz': generate_lorenz}

for name, fn in GENERATORS.items():
    tk, ct, _ = fn(5, seed=1991)
    recon = tk.astype(np.float32) / (N_BINS-1)
    print(f'{name:12s}: tokens={tk.shape} contin={ct.shape} max_quant_err={np.abs(ct-recon).max():.6f}')
print('Dual-return generators ready')

In [ ]:
# Perturbation Suite
from dataclasses import dataclass, field

@dataclass
class PerturbedSet:
    name: str; category: str; sequences: np.ndarray
    params: dict = field(default_factory=dict); description: str = ''

class ContinuousPerturbationSuite:
    def __init__(self, seed=SEED, noise_rates=None):
        self.rng = np.random.default_rng(seed)
        self.noise_rates = noise_rates or NOISE_RATES
    def run_all(self, seqs):
        results = {}
        for rate in self.noise_rates:
            name = f'noise_{int(rate*100)}pct'
            out = seqs.copy()
            mask = self.rng.random(out.shape) < rate
            noise = self.rng.normal(0, N_BINS*0.10, size=out.shape).astype(np.int64)
            out[mask] = np.clip(out[mask]+noise[mask], 0, N_BINS-1)
            results[name] = PerturbedSet(name=name, category='noise', sequences=out)
        results['time_reversal'] = PerturbedSet(name='time_reversal', category='reversal', sequences=seqs[:,::-1].copy())
        return results
print('Perturbation suite ready')

In [ ]:
# Models -- SmallMamba_Continuous + SmallBERT_Continuous
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

if MAMBA_AVAILABLE:
    from mamba_ssm import Mamba as MambaBlock
    class SmallMamba_Continuous(nn.Module):
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_layers=N_LAYERS,
                     d_state=D_STATE, d_conv=D_CONV, expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop = nn.Dropout(dropout)
            self.layers = nn.ModuleList([MambaBlock(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand) for _ in range(num_layers)])
            self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, 1)
        def forward(self, x, return_hidden=False):
            B, L = x.shape
            h = self.drop(self.tok_emb(x) + self.pos_emb(torch.arange(L, device=x.device).unsqueeze(0)))
            for layer, norm in zip(self.layers, self.norms): h = h + layer(norm(h))
            h = self.final_norm(h); out = self.head(h).squeeze(-1)
            return (out, h) if return_hidden else out
else:
    class SimpleSSMLayer(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__()
            d_inner = d_model * expand
            self.in_proj = nn.Linear(d_model, d_inner*2, bias=False)
            self.conv1d = nn.Conv1d(d_inner, d_inner, d_conv, padding=d_conv-1, groups=d_inner)
            self.D = nn.Parameter(torch.ones(d_inner))
            self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        def forward(self, x):
            B,L,D = x.shape; xz = self.in_proj(x); xi, z = xz.chunk(2, dim=-1)
            xc = self.conv1d(xi.transpose(1,2))[:,:,:L].transpose(1,2)
            return self.out_proj(torch.silu(xc) * torch.silu(z) * self.D.unsqueeze(0).unsqueeze(0))

    class SmallMamba_Continuous(nn.Module):
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_layers=N_LAYERS,
                     d_state=D_STATE, d_conv=D_CONV, expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop = nn.Dropout(dropout)
            self.layers = nn.ModuleList([SimpleSSMLayer(d_model, d_state, d_conv, expand) for _ in range(num_layers)])
            self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, 1)
        def forward(self, x, return_hidden=False):
            B, L = x.shape
            h = self.drop(self.tok_emb(x) + self.pos_emb(torch.arange(L, device=x.device).unsqueeze(0)))
            for layer, norm in zip(self.layers, self.norms): h = h + layer(norm(h))
            h = self.final_norm(h); out = self.head(h).squeeze(-1)
            return (out, h) if return_hidden else out

class SmallBERT_Continuous(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEADS,
                 num_layers=N_LAYERS, dim_feedforward=FFN_DIM, max_seq_len=SEQ_LEN, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        el = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                                        dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(el, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)
    def _causal_mask(self, L, device):
        return torch.triu(torch.ones(L,L,device=device), diagonal=1).masked_fill_(torch.triu(torch.ones(L,L,device=device), diagonal=1)==1, float('-inf'))
    def forward(self, x, return_hidden=False):
        B, L = x.shape
        h = self.drop(self.tok_emb(x) + self.pos_emb(torch.arange(L, device=x.device).unsqueeze(0)))
        h = self.norm(self.encoder(h, mask=self._causal_mask(L, x.device)))
        out = self.head(h).squeeze(-1)
        return (out, h) if return_hidden else out

# =============================================================================
# SmallStripedHyena (Evo 2 architecture: Hyena conv + Attention stripes)
# =============================================================================

class ImplicitFilterMLP(nn.Module):
    """Learns long convolution filter implicitly via MLP (Hyena's key innovation)."""
    def __init__(self, d_model, seq_len, n_hidden=64):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        n_pos_features = 16
        self.pos_emb = nn.Linear(n_pos_features, n_hidden)
        self.mlp = nn.Sequential(nn.GELU(), nn.Linear(n_hidden, n_hidden), nn.GELU(), nn.Linear(n_hidden, d_model))
        self.decay = nn.Parameter(torch.linspace(0.01, 5.0, d_model))
        self.register_buffer('pos_features', self._make_pos_features(seq_len, n_pos_features))

    def _make_pos_features(self, seq_len, n_features):
        positions = torch.linspace(0, 1, seq_len).unsqueeze(1)
        freqs = torch.arange(n_features).float() * math.pi
        return torch.sin(positions * freqs.unsqueeze(0))

    def forward(self, seq_len):
        if seq_len != self.seq_len:
            pf = self._make_pos_features(seq_len, self.pos_features.shape[1]).to(self.pos_features.device)
        else:
            pf = self.pos_features
        h = self.mlp(self.pos_emb(pf))
        t = torch.linspace(0, 1, seq_len, device=h.device).unsqueeze(1)
        return (h * torch.exp(-self.decay.unsqueeze(0) * t)).T


class HyenaOperator(nn.Module):
    """Data-controlled long convolution with gating (replaces attention)."""
    def __init__(self, d_model, seq_len, order=2, short_conv_kernel=3):
        super().__init__()
        self.d_model, self.order = d_model, order
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.short_convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, kernel_size=short_conv_kernel,
                      padding=short_conv_kernel // 2, groups=d_model) for _ in range(order + 1)])
        self.filters = nn.ModuleList([ImplicitFilterMLP(d_model, seq_len) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model)

    def _fft_conv(self, signal, kernel):
        L = signal.shape[-1]
        fft_len = 2 * L
        return torch.fft.irfft(
            torch.fft.rfft(signal, n=fft_len, dim=-1) * torch.fft.rfft(kernel, n=fft_len, dim=-1).unsqueeze(0),
            n=fft_len, dim=-1)[..., :L]

    def forward(self, x):
        B, L, D = x.shape
        branches = self.in_proj(x).reshape(B, L, self.order + 1, D)
        conv_outputs = [self.short_convs[i](branches[:, :, i, :].transpose(1, 2)) for i in range(self.order + 1)]
        v = conv_outputs[0]
        for i in range(self.order):
            v = self._fft_conv(v, self.filters[i](L)) * conv_outputs[i + 1]
        return self.out_proj(v.transpose(1, 2))


class SHMultiHeadAttention(nn.Module):
    """Standard multi-head attention with RoPE (for StripedHyena minority layers)."""
    def __init__(self, d_model, n_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('freqs', torch.outer(torch.arange(max_seq_len).float(), freqs))

    def _apply_rotary(self, x, freqs):
        L = x.shape[2]
        freqs = freqs[:L]
        cos_f = torch.cos(freqs).unsqueeze(0).unsqueeze(0)
        sin_f = torch.sin(freqs).unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack([x1 * cos_f - x2 * sin_f, x1 * sin_f + x2 * cos_f], dim=-1).flatten(-2)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        q, k = self._apply_rotary(q, self.freqs), self._apply_rotary(k, self.freqs)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        attn = F.softmax(attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf')), dim=-1)
        return self.out_proj((attn @ v).transpose(1, 2).reshape(B, L, D))


class StripedHyenaBlock(nn.Module):
    """Single StripedHyena block: Hyena or Attention + SwiGLU MLP."""
    def __init__(self, d_model, seq_len, n_heads=4, order=2, is_attention=False, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mixer = SHMultiHeadAttention(d_model, n_heads) if is_attention else HyenaOperator(d_model, seq_len, order=order)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp_gate = nn.Linear(d_model, mlp_hidden)
        self.mlp_value = nn.Linear(d_model, mlp_hidden)
        self.mlp_out = nn.Linear(mlp_hidden, d_model)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        h = self.norm2(x)
        x = x + self.mlp_out(F.silu(self.mlp_gate(h)) * self.mlp_value(h))
        return x


class SmallStripedHyena(nn.Module):
    """
    Minimal StripedHyena (Evo 2 arch) for bottleneck isolation.
    ~3.4M params: d_model=128, 8 layers (6 Hyena + 2 Attention), seq_len=512.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                 n_heads=N_HEADS, seq_len=SEQ_LEN, order=2, mlp_ratio=4,
                 attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model, self.vocab_size = d_model, vocab_size
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order, is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        # No weight tying (vocab_size=258 != d_model=256)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        logits = self.head(h)
        return (logits, h) if return_hidden else logits


class SmallStripedHyena_Continuous(nn.Module):
    """SmallStripedHyena with continuous scalar output head (MSE loss)."""
    def __init__(self, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
                 seq_len=SEQ_LEN, order=2, mlp_ratio=4, attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        # Use token embeddings (same as BERT/Mamba continuous variants)
        self.tok_emb = nn.Embedding(VOCAB_SIZE, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order, is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena_Continuous: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.drop(self.tok_emb(x))
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        out = self.head(h).squeeze(-1)
        return (out, h) if return_hidden else out



MODEL_CLASSES = {'SmallBERT_Continuous': SmallBERT_Continuous, 'SmallMamba_Continuous': SmallMamba_Continuous, 'SmallStripedHyena_Continuous': SmallStripedHyena_Continuous}
ARCHITECTURES = list(MODEL_CLASSES.keys())
for name, cls in MODEL_CLASSES.items():
    m = cls(); print(f'{name:30s}: {sum(p.numel() for p in m.parameters())/1e6:.2f}M'); del m
print('Models ready (both with continuous MSE head)')


In [ ]:
# Training Loop (MSE on raw continuous ODE targets)
from torch.utils.data import DataLoader, TensorDataset

def train_model_continuous(model, train_tokens, train_contin, val_tokens, val_contin,
                           arch_name, ds_name, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED):
    """Train with MSE on raw continuous targets (NOT requantized staircase)."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = model.to(DEVICE)
    tl = DataLoader(TensorDataset(torch.from_numpy(train_tokens).long(),
                                   torch.from_numpy(train_contin).float()),
                    batch_size=batch_size, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
    vl = DataLoader(TensorDataset(torch.from_numpy(val_tokens).long(),
                                   torch.from_numpy(val_contin).float()),
                    batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs*len(tl))
    criterion = nn.MSELoss()
    best_val, best_state = float('inf'), None
    ckpt = f'{CKPT_DIR}/{arch_name}_{ds_name}_seed{seed}_best.pt'
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        print(f'  Loaded checkpoint: {ckpt}'); return model, [], []
    print(f'  Training {arch_name} on {ds_name} (MSE on raw ODE floats, {epochs} epochs)...')
    t0 = time.time(); train_losses, val_losses = [], []
    for ep in range(epochs):
        model.train(); eloss, nb = 0, 0
        for b_tok, b_cont in tl:
            b_tok, b_cont = b_tok.to(DEVICE), b_cont.to(DEVICE)
            inputs = b_tok[:, :-1]      # discrete token input
            targets = b_cont[:, 1:]     # RAW continuous target
            loss = criterion(model(inputs), targets)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); eloss += loss.item(); nb += 1
        train_losses.append(eloss/nb)
        model.eval(); vloss, vb = 0, 0
        with torch.no_grad():
            for b_tok, b_cont in vl:
                b_tok, b_cont = b_tok.to(DEVICE), b_cont.to(DEVICE)
                vloss += criterion(model(b_tok[:,:-1]), b_cont[:,1:]).item(); vb += 1
        avg_v = vloss/max(vb,1); val_losses.append(avg_v)
        if avg_v < best_val: best_val = avg_v; best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        if (ep+1)%5==0 or ep==0: print(f'    Ep {ep+1:3d}/{epochs} train={eloss/nb:.6f} val={avg_v:.6f} [{time.time()-t0:.0f}s]')
    if best_state: model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt)
    print(f'  Done in {(time.time()-t0)/60:.1f}min (best MSE: {best_val:.6f})')
    return model, train_losses, val_losses
print('Training loop ready (MSE on raw continuous ODE targets)')

In [ ]:
# Evaluation + Main Experiment Loop
from evaluation_harness import StabilityHarness
harness = StabilityHarness(window_size=0, metric='cosine', n_splits=30, seed=320, max_samples=2500, n_bootstrap=5)

def cleanup(): gc.collect(); torch.cuda.is_available() and torch.cuda.empty_cache()

@torch.no_grad()
def extract_embeddings(model, seqs, bs=128):
    model.eval(); embs = []
    for i in range(0,len(seqs),bs):
        _, h = model(torch.from_numpy(seqs[i:i+bs]).long().to(DEVICE), return_hidden=True)
        embs.append(h.mean(dim=1).cpu().numpy())
    return np.concatenate(embs)

print('='*70); print('ABLATION 2C: Mamba vs BERT with Continuous MSE Head (raw ODE targets)'); print('='*70)
all_rows = []
for ds in DATASETS:
    print(f"\n{'='*70}\nDATASET: {ds.upper()}\n{'='*70}")
    train_tokens, train_contin, grange = GENERATORS[ds](N_TRAIN, seed=SEED)
    GLOBAL_RANGES[ds] = grange
    eval_tokens, eval_contin, _ = GENERATORS[ds](N_EVAL, seed=SEED+1000, global_range=grange)
    perts = ContinuousPerturbationSuite(seed=SEED).run_all(eval_tokens)
    for arch in ARCHITECTURES:
        print(f'\n  --- {arch} ---')
        ckey = f'{arch}_{ds}'
        model = MODEL_CLASSES[arch]()
        np_m = sum(p.numel() for p in model.parameters())/1e6
        model, _, _ = train_model_continuous(model, train_tokens, train_contin,
                                            eval_tokens, eval_contin, arch, ds)
        cc = f'{CACHE_DIR}/{ckey}_clean.npy'
        emb_clean = np.load(cc) if os.path.exists(cc) else extract_embeddings(model, eval_tokens)
        if not os.path.exists(cc): np.save(cc, emb_clean)
        pembs = {}
        for pn, ps in perts.items():
            cp = f'{CACHE_DIR}/{ckey}_{pn}.npy'
            pembs[pn] = np.load(cp) if os.path.exists(cp) else extract_embeddings(model, ps.sequences)
            if not os.path.exists(cp): np.save(cp, pembs[pn])
        report = harness.evaluate_all_perturbations(model_name=ckey, embeddings_clean=emb_clean, perturbed_dict=pembs)
        for pn, m in report.perturbation_breakdown().items():
            all_rows.append({'dataset':ds, 'architecture':arch, 'variant':'Ablation2C_Continuous',
                'n_params_M':round(np_m,2), 'perturbation':pn,
                'rdm_similarity':m.get('rdm_similarity_score',0), 'composite':m.get('composite_stability',0)})
        s = report.summary()
        print(f'    Composite: {s["mean_composite_stability"]:.4f}')
        del model; cleanup()

df = pd.DataFrame(all_rows)
p = f'{RESULTS_DIR}/variant_c_mamba_continuous_detailed.csv'
df.to_csv(p, index=False); print(f'\nSaved: {p}')
print(df.to_string(index=False))

In [ ]:
# Visualization
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11})

# Load baseline + Variant A for comparison
frames = [df]
for tag, path in [('Baseline', './results/architectural_control_experiment/results/architectural_control_detailed_v2.csv'),
                   ('VariantA', './results/ablation_variant_a_continuous/results/variant_a_continuous_detailed.csv')]:
    if os.path.exists(path):
        tmp = pd.read_csv(path); tmp['variant'] = tag; frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(12, 6))
agg = df_all.groupby(['variant', 'architecture'])['composite'].agg(['mean','std']).reset_index()
colors = {'SmallBERT':'#2563EB','SmallMamba':'#16A34A','SmallBERT_Continuous':'#DC2626','SmallMamba_Continuous':'#059669'}

x = np.arange(len(agg))
bars = ax.bar(x, agg['mean'], yerr=agg['std'], capsize=4,
              color=[colors.get(a, '#6B7280') for a in agg['architecture']], alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"{r['architecture']}\n({r['variant']})" for _,r in agg.iterrows()], fontsize=8, rotation=30, ha='right')
for bar, val in zip(bars, agg['mean']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')
ax.set_ylim(0, 1.05); ax.set_ylabel('Composite Stability')
ax.set_title('Variant C: Mamba Stays Stable With Continuous Head (Raw ODE Targets)', fontweight='bold')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
fp = f'{RESULTS_DIR}/variant_c_comparison.png'
plt.savefig(fp, dpi=300, bbox_inches='tight'); plt.show()
print(f'Saved: {fp}')